Dataset: https://www.kaggle.com/datasets/matinmahmoudi/loans-and-liability/data

In [ ]:
# Import tools
import pandas as pd
from sklearn.naive_bayes import GaussianNB         # NB for continuous numbers
from sklearn.model_selection import train_test_split  # Split data into train/test
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report  # Check how well we did
import numpy as np

In [ ]:
# load dataset
df = pd.read_csv("LoanData_Preprocessed_v1.2.csv")
df.head()

,age,employ,address,income,debtinc,creddebt,othdebt,ed,default
0,29,11,7,32.0,6.0,0.927360,0.992640,1,0
1,28,1,3,26.0,12.4,0.377208,2.846792,4,0
2,34,16,3,75.0,10.4,3.954600,3.845400,1,0
3,51,31,14,249.0,7.8,4.272840,15.149160,2,0
4,40,13,11,102.0,18.9,6.226794,13.051206,2,1


In [ ]:
# dataset overview
df.info()
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 485 entries, 0 to 484
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       485 non-null    int64  
 1   employ    485 non-null    int64  
 2   address   485 non-null    int64  
 3   income    485 non-null    float64
 4   debtinc   485 non-null    float64
 5   creddebt  485 non-null    float64
 6   othdebt   485 non-null    float64
 7   ed        485 non-null    int64  
 8   default   485 non-null    int64  
dtypes: float64(4), int64(5)
memory usage: 34.2 KB
(485, 9)


In [ ]:
# view output label distribution
print(df['default'].value_counts()) #  1 indicates default and 0 indicates no default

default
0    364
1    121
Name: count, dtype: int64


In [ ]:
# define the X features and y (output label)
X = df.drop(["default"], axis=1)  # Input features
y = df["default"]                 # Output label

In [ ]:
# split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42,      # Same split every run (reproducible)
    stratify=y            # Keeps class distribution the same in train and test sets
)

In [ ]:
# Train dataset with GaussianNB
model = GaussianNB()         # Create an empty model
model.fit(X_train, y_train)  # Train it on training set


GaussianNB()

In [ ]:
# Evaluate model
y_predicted = model.predict(X_test)   # Model guesses each customer outcome

accuracy = accuracy_score(y_test, y_predicted)   # % of correct guesses
print(f"Accuracy: {accuracy * 100:.1f}%")
print(f"(Got {int(accuracy * len(y_test))} out of {len(y_test)} consumer default / no default outcomes correct)")
print()


# Classification Report for model
print("\nClassification Report:\n", classification_report(y_test, y_predicted))   # display metrics to evaluate model

Accuracy: 82.5%
(Got 80 out of 97 consumer default / no default outcomes correct)


Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.90      0.89        73
           1       0.67      0.58      0.62        24

    accuracy                           0.82        97
   macro avg       0.77      0.74      0.75        97
weighted avg       0.82      0.82      0.82        97



In [ ]:
# confusion matrix
cm = confusion_matrix(y_test, y_predicted)

print("=== Confusion Matrix ===")
print("Rows = actual outcome, Columns = predicted outcome")
print(f"{'':15} {'No Default':>10} {'Default':>12}")
print("-" * 50)
labels = ["No Default", "Default"]
for i, row in enumerate(cm):
    print(f"{labels[i]:15} {row[0]:>10} {row[1]:>12}")
print()

=== Confusion Matrix ===
Rows = actual outcome, Columns = predicted outcome
                No Default      Default
--------------------------------------------------
No Default              66            7
Default                 10           14



In [ ]:
print("= Predict a brand new customer ===")
print()

# A new customer
new_customer = [[40, 12, 12, 65.0, 18.5, 8.2, 4.5, 4]]   # [age, employ, address, income, debtinc, creddebt, othdebt, ed]

# Get prediction
prediction = model.predict(new_customer)[0]                  # 0 or 1
probabilities = model.predict_proba(new_customer)[0]         # Probability for each of the two outcomes

print(f"New customer data:")
print(f"  Age: {new_customer[0][0]} ")
print(f"  Income : {new_customer[0][3]} ")
print(f"  Employed Years: {new_customer[0][1]} ")
print(f"  Years living in same address: {new_customer[0][2]}")
print(f"  debt-to-income ratio: {new_customer[0][4]}")
print(f"  Amount of credit card debt the applicant has: {new_customer[0][5]}")
print(f"  Amount of other debts the applicant has: {new_customer[0][6]}")


print()
print("Prediction:", "Default" if prediction == 1 else "No Default")
print()
print(f"Probabilities:")
for i, name in enumerate(model.classes_):
    bar = "#" * int(probabilities[i] * 30)    # Simple text bar chart
    print(f"  {name:12} | {bar:<30} | {probabilities[i]*100:.1f}%")

= Predict a brand new customer ===

New customer data:
  Age: 40 
  Income : 65.0 
  Employed Years: 12 
  Years living in same address: 12
  debt-to-income ratio: 18.5
  Amount of credit card debt the applicant has: 8.2
  Amount of other debts the applicant has: 4.5

Prediction: Default

Probabilities:
             0 |                                | 0.4%
             1 | #############################  | 99.6%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GaussianNB was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GaussianNB was fitted with feature names
  warnings.warn(


The loans and liability dataset was used to build a model to predict whether a customer is likely to default on loan. The model used various financial metrics such as age, income, previous financial history, and other important indicators like employment years. Due to continuous and numerical features, the GaussianNB was well-suited for the dataset. There was one categorical feature, level of education, which was processed into numerical values for model processing. The dataset was split and trained, and predicted on the test set. The model showed 82.5% accuracy. Further evaluation of individual class performance, shows it is stronger in predicting default (0) cases compared to default (1). The model misses about 40% of actual default cases, and is inaccurate for 33% of the time when predicting default. This can be significant since it is missing out in identifying customers that would likely default and instead classifying them as no default. Overall, closer evaluation showed an imbalance in class performance that leads to poor predictions of the critical customers that might default on a loan.     
